# Imputação de `zona_emb` nos domingos

A entrega revisada de domingos (`SEI! 5010.2026-0008307-9`) traz `zona_emb = 999` (zona não
classificada) em **praticamente 100% dos registros**, nos dois anos — defeito confirmado nos
arquivos brutos, não introduzido pelo pipeline. Sem isso, o domingo — que é exatamente o dia da
Tarifa Zero — não tem análise por zona, e as Seções 3, 5 e 6 de
`03_comparacoes/compara_demanda_oferta_genero_idade.ipynb` precisam usar `linha_blt` como unidade.

`04_diagnostico_domingos_antiga_vs_nova.ipynb` estabeleceu o que torna a correção possível: a
entrega antiga (`Pedido 097465`) é **a mesma população de eventos**, com registros replicados por
um fator inteiro que varia por classe de pagamento (2x em VT/IDOSO/PCD, ~1,12x em COMUM), e
**cobre 100% dos pares (hash, linha)** da entrega nova. Ela tem `zona_emb` real (387 zonas).
Logo, serve como **doadora do atributo `zona_emb`** — nunca das suas contagens.

## Método: alocação proporcional, não moda

Para cada par `(hash_anonimizado, linha_blt)`, a entrega antiga dá uma *distribuição* de zonas
(uma pessoa pode embarcar na mesma linha em zonas diferentes no mesmo domingo — ida e volta de
pontos distintos). Como a antiga é a nova multiplicada por um fator constante dentro da classe,
essa distribuição é, a menos de arredondamento, a distribuição verdadeira dos embarques da nova.

Então em vez de escolher a zona modal (que jogaria 100% do peso numa zona e enviesaria o mapa),
os embarques de cada par são **alocados proporcionalmente** às frequências do doador:

```
w(zona | hash, linha) = k_antiga(hash, linha, zona) / Σ_zona k_antiga(hash, linha, zona)
N_embarques(zona)     = n_nova(hash, linha, ...) × w(zona | hash, linha)
```

Consequências, ambas deliberadas:

- **`N_embarques` sai fracionário.** É uma estimativa de alocação, não uma contagem — arredondar
  só na hora de reportar. A soma por data continua **exatamente** igual à contagem bruta da base
  nova (invariante conferido célula a célula abaixo).
- **O fator de replicação da antiga não entra em lugar nenhum**, porque só o peso relativo dentro
  do par é usado. É isso que separa "doar atributo" de "doar volume".

## Limites conhecidos

- **`zona_emb` já é uma estimativa da própria SPTrans** (GPS do ônibus × endereço cadastrado ×
  horário da validação), não uma observação. A imputação herda esse erro e acrescenta o seu.
- **Registros sem `linha_blt` (~25%) não recebem zona** — não têm chave. Pelo `04`, esses não são
  embarque de ônibus municipal (provavelmente Metrô/CPTM com Bilhete Único) e devem sair de
  qualquer contagem de demanda de ônibus de todo modo.
- **Em `DINHEIRO` e `OUTROS` o `hash` não identifica uma pessoa** (a SPTrans adverte: numa amostra,
  781 mil registros `DINHEIRO` compartilham 9.637 hashes). A chave do par ali agrega gente
  diferente, então a zona imputada é bem mais frágil. A coluna `classificacao` fica na saída
  justamente para permitir filtrar `CLASSES_BU` nas análises que exigem rigor.
- **Pares ambíguos** (mais de uma zona no doador) respondem por ~28% do volume imputado. A
  alocação proporcional é o tratamento certo para eles, mas é uma distribuição estimada, não a
  zona de cada embarque individual.

Saída: `../outputs/01/Dados_Domingos_2023_2024_zona_imputada.parquet`
(`data × linha_blt × zona_emb × classificacao × genero × faixa_etaria → N_embarques`).

## Seção 0 — Constantes

In [1]:
import glob
import os
import re
import time

import duckdb
import pandas as pd

# Raiz das fontes brutas (fora do repo). Se a unidade mudar, muda so esta linha.
RAIZ_SPTRANS = r"C:\Users\9837292\Desktop\SSD\SPTrans"

PASTA_ANTIGA = os.path.join(RAIZ_SPTRANS, "Pedido 097465 - dados de bilhetagem", "arquivos PARQUET")
PASTA_NOVA = os.path.join(RAIZ_SPTRANS, "SEI! 5010.2026-0008307-9")

# Convenção do repositório: outputs vão para ../outputs/<NN>/, onde NN é o prefixo da
# pasta do notebook que os gerou (este está em 01_criacao_de_bases/).
PASTA_SAIDA = "../outputs/01"
CAMINHO_SAIDA = f"{PASTA_SAIDA}/Dados_Domingos_2023_2024_zona_imputada.parquet"
CAMINHO_4MESES = f"{PASTA_SAIDA}/Dados_4Meses_2023_2024.parquet"

ZONA_SENTINELA = 999   # "zona nao classificada" na entrega nova
FORCE_RECOMPUTE = False

os.makedirs(PASTA_SAIDA, exist_ok=True)
pd.set_option("display.width", 200)

con = duckdb.connect()
con.execute(f"PRAGMA threads={max(1, (os.cpu_count() or 4) - 2)}")
con.execute("PRAGMA memory_limit='32GB'")

## Seção 1 — Inventário das duas entregas

A imputação só existe para as datas presentes **nas duas**. O `04` já verificou que são as mesmas
101 datas, mas a checagem fica aqui porque é ela que define o universo processado.

In [2]:
def datas_de(pasta):
    """{AAAAMMDD: caminho} dos parquets de uma entrega, pela data no nome do arquivo."""
    out = {}
    for f in sorted(glob.glob(os.path.join(pasta, "*.parquet"))):
        m = re.search(r"20\d{6}", os.path.basename(f))
        if m:
            out[m.group(0)] = f
    return out


antiga, nova = datas_de(PASTA_ANTIGA), datas_de(PASTA_NOVA)
datas = sorted(set(antiga) & set(nova))

print(f"antiga: {len(antiga)} datas | nova: {len(nova)} datas | em ambas: {len(datas)}")
print("so na nova  :", sorted(set(nova) - set(antiga)))
print("so na antiga:", sorted(set(antiga) - set(nova)))

antiga: 101 datas | nova: 101 datas | em ambas: 101
so na nova  : []
so na antiga: []


## Seção 2 — Imputação

Uma consulta DuckDB por data (~1 s cada). Duas coisas conferidas em cada iteração:

1. **Invariante de volume**: `Σ N_embarques` tem que ser *exatamente* o número de registros da
   base nova naquela data. O join transporta atributo; se criar ou perder embarque, é bug.
2. **Cobertura**: quanto do volume (e quanto do volume *com linha*) recebeu zona.

In [3]:
SQL_IMPUTA = """
WITH doador AS (         -- distribuicao de zonas do par (hash, linha) na entrega antiga
    SELECT hash_anonimizado, linha_blt, zona_emb, count(*)::DOUBLE AS k
    FROM read_parquet(?)
    WHERE linha_blt IS NOT NULL AND zona_emb IS NOT NULL AND zona_emb <> {sentinela}
    GROUP BY 1, 2, 3
),
peso AS (                -- peso relativo dentro do par: o fator de replicacao da antiga cancela
    SELECT hash_anonimizado, linha_blt, zona_emb,
           k / sum(k) OVER (PARTITION BY hash_anonimizado, linha_blt) AS w,
           count(*) OVER (PARTITION BY hash_anonimizado, linha_blt) AS nz
    FROM doador
),
receptor AS (            -- entrega nova, ja contada por par x atributos
    SELECT data, hash_anonimizado, linha_blt, classificacao, genero, faixa_etaria,
           count(*)::DOUBLE AS n
    FROM read_parquet(?)
    GROUP BY 1, 2, 3, 4, 5, 6
)
SELECT r.data, r.linha_blt, p.zona_emb, r.classificacao, r.genero, r.faixa_etaria,
       sum(r.n * coalesce(p.w, 1.0)) AS N_embarques,
       sum(CASE WHEN p.nz > 1 THEN r.n * p.w ELSE 0 END) AS N_par_ambiguo
FROM receptor r
LEFT JOIN peso p USING (hash_anonimizado, linha_blt)
GROUP BY 1, 2, 3, 4, 5, 6
""".format(sentinela=ZONA_SENTINELA)


def imputa_data(d):
    """Agregado com zona imputada para uma data, + linha de diagnostico."""
    parte = con.execute(SQL_IMPUTA, [[antiga[d]], [nova[d]]]).df()
    bruto = con.execute("SELECT count(*) FROM read_parquet(?)", [[nova[d]]]).fetchone()[0]

    tem_zona = parte["zona_emb"].notna()
    tem_linha = parte["linha_blt"].notna()
    diag = {
        "data": d,
        "registros_nova": bruto,
        "soma_imputada": parte["N_embarques"].sum(),
        "vol_com_linha": parte.loc[tem_linha, "N_embarques"].sum(),
        "vol_com_zona": parte.loc[tem_zona, "N_embarques"].sum(),
        "vol_par_ambiguo": parte["N_par_ambiguo"].sum(),
        "n_zonas": parte["zona_emb"].nunique(),
    }
    return parte.drop(columns=["N_par_ambiguo"]), diag


if not FORCE_RECOMPUTE and os.path.exists(CAMINHO_SAIDA):
    imputado = pd.read_parquet(CAMINHO_SAIDA)
    diagnostico = None
    print(f"Carregado do cache: {CAMINHO_SAIDA} ({len(imputado):,} linhas). "
          f"Defina FORCE_RECOMPUTE=True para recalcular.")
else:
    t0 = time.time()
    partes, diags = [], []
    for i, d in enumerate(datas, 1):
        parte, diag = imputa_data(d)
        # Invariante duro: nao pode criar nem perder embarque.
        assert abs(diag["soma_imputada"] - diag["registros_nova"]) < 1e-6, (d, diag)
        partes.append(parte)
        diags.append(diag)
        if i % 20 == 0:
            print(f"  {i}/{len(datas)} ({time.time() - t0:.0f}s)", flush=True)

    imputado = pd.concat(partes, ignore_index=True)
    diagnostico = pd.DataFrame(diags)
    imputado.to_parquet(CAMINHO_SAIDA, index=False)
    print(f"\nSalvo em {CAMINHO_SAIDA} em {time.time() - t0:.0f}s")

print(f"\n{len(imputado):,} linhas | {imputado['data'].nunique()} datas | "
      f"{imputado['zona_emb'].nunique()} zonas")
imputado.head(3)

  20/101 (11s)


  40/101 (22s)


  60/101 (32s)


  80/101 (44s)


  100/101 (56s)



Salvo em ../outputs/01/Dados_Domingos_2023_2024_zona_imputada.parquet em 65s



25,137,548 linhas | 101 datas | 355 zonas


,data,linha_blt,zona_emb,classificacao,genero,faixa_etaria,N_embarques
0,20230101,9653-10,37.0,VT,F,30 a 39,4.0
1,20230101,3746-10,261.0,VT,M,20 a 29,0.8
2,20230101,N732-11,308.0,COMUM,F,50 a 59,2.0


## Seção 3 — Cobertura e ambiguidade

Três números importam, e valem para o volume (embarques), não para a contagem de linhas do
agregado:

- **cobertura sobre o total** — limitada por construção, porque ~25% dos registros não têm linha;
- **cobertura sobre o volume com linha** — a métrica que de fato mede a imputação;
- **fração vinda de par ambíguo** — o quanto do resultado depende da alocação proporcional em vez
  de uma zona única e inequívoca do doador.

In [4]:
if diagnostico is None:   # recarregado do cache: recompoe o diagnostico do proprio agregado
    diagnostico = (imputado.assign(
        vol_com_linha=lambda d: d["N_embarques"].where(d["linha_blt"].notna(), 0),
        vol_com_zona=lambda d: d["N_embarques"].where(d["zona_emb"].notna(), 0))
        .groupby("data")
        .agg(soma_imputada=("N_embarques", "sum"),
             vol_com_linha=("vol_com_linha", "sum"),
             vol_com_zona=("vol_com_zona", "sum"),
             n_zonas=("zona_emb", "nunique"))
        .reset_index())
    diagnostico["registros_nova"] = diagnostico["soma_imputada"]

diagnostico["Ano"] = diagnostico["data"].str[:4]
diagnostico["pct_zona_total"] = 100 * diagnostico["vol_com_zona"] / diagnostico["soma_imputada"]
diagnostico["pct_zona_com_linha"] = 100 * diagnostico["vol_com_zona"] / diagnostico["vol_com_linha"]

cols = ["pct_zona_total", "pct_zona_com_linha", "n_zonas"]
if "vol_par_ambiguo" in diagnostico:
    diagnostico["pct_ambiguo"] = 100 * diagnostico["vol_par_ambiguo"] / diagnostico["vol_com_zona"]
    cols.append("pct_ambiguo")

print("por ano (media entre os domingos):")
print(diagnostico.groupby("Ano")[cols].mean().round(2).to_string())

print("\npiores 5 domingos por cobertura (volume com linha):")
print(diagnostico.nsmallest(5, "pct_zona_com_linha")[["data"] + cols].round(2).to_string(index=False))

# Cobertura por classe de pagamento: em DINHEIRO/OUTROS o hash nao identifica pessoa,
# entao a zona ali e bem mais fragil, mesmo quando a cobertura e alta.
por_classe = imputado.assign(
    com_zona=lambda d: d["N_embarques"].where(d["zona_emb"].notna(), 0)).groupby("classificacao").agg(
    volume=("N_embarques", "sum"), com_zona=("com_zona", "sum"))
por_classe["pct_com_zona"] = 100 * por_classe["com_zona"] / por_classe["volume"]
print("\ncobertura por classificacao:")
print(por_classe.sort_values("volume", ascending=False).round(1).to_string())

por ano (media entre os domingos):
      pct_zona_total  pct_zona_com_linha  n_zonas  pct_ambiguo
Ano                                                           
2023           73.73               99.15   354.83        26.00
2024           78.03               99.47   354.96        32.53

piores 5 domingos por cobertura (volume com linha):
    data  pct_zona_total  pct_zona_com_linha  n_zonas  pct_ambiguo
20230108           74.14               98.07      354        27.62
20230910           72.55               98.65      355        26.46
20230507           74.01               98.67      355        27.00
20230326           74.38               98.79      354        27.91
20240714           77.75               98.80      355        32.67



cobertura por classificacao:
                    volume     com_zona  pct_com_zona
classificacao                                        
COMUM          163650072.0  133749039.0          81.7
VT              67878961.0   42780312.0          63.0
IDOSO           33842267.0   25460059.0          75.2
DINHEIRO        26799367.0   26450662.0          98.7
PCD             14269641.0   11269102.0          79.0
ESTUDANTE       11299505.0    5777782.0          51.1
OUTROS           4731088.0     308416.0           6.5


## Seção 4 — Sanidade espacial

A imputação pode ser internamente consistente e ainda assim estar errada. O teste externo
disponível: comparar, **por linha**, a distribuição de embarques entre zonas no domingo imputado
com a do sábado da base de 4-meses — que tem `zona_emb` original e não passou por imputação
nenhuma. Sábado não é domingo (o perfil de viagem muda), então não se espera identidade; espera-se
que as linhas embarquem *nas mesmas zonas*, com pesos parecidos.

A métrica é a **similaridade de cosseno** entre os dois vetores de participação por zona, por
linha, ponderada pelo volume da linha. Se a imputação estivesse embaralhando zonas, essa
similaridade cairia para perto de zero.

In [5]:
MESES = ["04", "05", "09", "10"]   # janela comum as duas bases

# Sabado da base de 4-meses: zona_emb original, sem imputacao.
sabado = con.execute("""
    SELECT linha_blt, zona_emb, count(*)::DOUBLE AS n
    FROM read_parquet(?)
    WHERE linha_blt IS NOT NULL
      AND zona_emb IS NOT NULL AND zona_emb <> {sentinela}
      AND substr(data, 5, 2) IN ('04','05','09','10')
      AND dayofweek(strptime(data, '%Y%m%d')) = 6      -- 6 = sabado no DuckDB
    GROUP BY 1, 2
""".format(sentinela=ZONA_SENTINELA), [[CAMINHO_4MESES]]).df()

domingo = (imputado[imputado["data"].str[4:6].isin(MESES)
                    & imputado["linha_blt"].notna() & imputado["zona_emb"].notna()]
           .groupby(["linha_blt", "zona_emb"], as_index=False)["N_embarques"].sum()
           .rename(columns={"N_embarques": "n"}))

print(f"sabado: {len(sabado):,} pares linha-zona | domingo imputado: {len(domingo):,}")


def cosseno_por_linha(a, b):
    """Similaridade de cosseno entre as distribuicoes de zona de cada linha."""
    j = a.merge(b, on=["linha_blt", "zona_emb"], how="outer", suffixes=("_a", "_b")).fillna(0)
    g = j.groupby("linha_blt")
    num = g.apply(lambda x: (x["n_a"] * x["n_b"]).sum(), include_groups=False)
    den = (g["n_a"].apply(lambda s: (s ** 2).sum() ** 0.5)
           * g["n_b"].apply(lambda s: (s ** 2).sum() ** 0.5))
    cos = (num / den).replace([float("inf")], float("nan")).dropna()
    vol = g["n_b"].sum().reindex(cos.index)
    return cos, vol


cos, vol = cosseno_por_linha(sabado, domingo)
media_ponderada = (cos * vol).sum() / vol.sum()

print(f"\nlinhas comparadas: {len(cos)}")
print(f"cosseno medio ponderado por volume de domingo: {media_ponderada:.3f}")
print(f"cosseno mediano (simples): {cos.median():.3f}")
print("\ndistribuicao do cosseno:")
print(cos.describe(percentiles=[.05, .25, .5, .75, .95]).round(3).to_string())

print("\n10 linhas com pior similaridade (entre as de maior volume):")
grandes = vol.nlargest(300).index
print(cos.reindex(grandes).nsmallest(10).round(3).to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

sabado: 14,574 pares linha-zona | domingo imputado: 13,378



linhas comparadas: 1149
cosseno medio ponderado por volume de domingo: 0.994
cosseno mediano (simples): 0.996

distribuicao do cosseno:
count    1149.000
mean        0.984
std         0.065
min         0.000
5%          0.948
25%         0.989
50%         0.996
75%         0.999
95%         1.000
max         1.000

10 linhas com pior similaridade (entre as de maior volume):
linha_blt
3098-10    0.965
5178-10    0.969
208V-10    0.971
888P-10    0.971
576M-10    0.977
2552-10    0.977
5110-10    0.978
175T-10    0.980
6049-10    0.981
1178-10    0.982


### Referência: o mesmo cosseno entre dois dias que ninguém imputou

O número acima só quer dizer alguma coisa contra uma linha de base. Sábado x dia útil, ambos com
`zona_emb` original, mede quanto a distribuição espacial de uma linha varia entre tipos de dia
**sem** imputação nenhuma. Se domingo-imputado x sábado ficar na mesma faixa, a imputação não está
introduzindo desordem além da variação natural entre dias.

In [6]:
util = con.execute("""
    SELECT linha_blt, zona_emb, count(*)::DOUBLE AS n
    FROM read_parquet(?)
    WHERE linha_blt IS NOT NULL
      AND zona_emb IS NOT NULL AND zona_emb <> {sentinela}
      AND substr(data, 5, 2) IN ('04','05','09','10')
      AND dayofweek(strptime(data, '%Y%m%d')) BETWEEN 1 AND 5
    GROUP BY 1, 2
""".format(sentinela=ZONA_SENTINELA), [[CAMINHO_4MESES]]).df()

cos_ref, vol_ref = cosseno_por_linha(sabado, util)
print(f"REFERENCIA sabado x util (sem imputacao): "
      f"ponderado={((cos_ref * vol_ref).sum() / vol_ref.sum()):.3f} | mediano={cos_ref.median():.3f}")
print(f"MEDIDO     sabado x domingo imputado:     "
      f"ponderado={media_ponderada:.3f} | mediano={cos.median():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

REFERENCIA sabado x util (sem imputacao): ponderado=0.987 | mediano=0.995
MEDIDO     sabado x domingo imputado:     ponderado=0.994 | mediano=0.996


## Seção 5 — Como consumir

`03_comparacoes/compara_demanda_oferta_genero_idade.ipynb` lê este arquivo como o lado *domingo*
do seu agregado de demanda (o lado 4-meses continua saindo da base diária). Duas regras ao usar:

1. **Filtrar `linha_blt` não nulo** em qualquer contagem de demanda de ônibus — os nulos não têm
   zona nem são embarque de ônibus municipal (`04_diagnostico`).
2. **`N_embarques` é fracionário**; some primeiro, arredonde depois.

E a advertência que segue valendo para qualquer número por zona publicado: `zona_emb` é uma
estimativa da SPTrans mesmo quando vem original, e aqui ela vem imputada por cima disso.